In [1]:
import pandas as pd
import boto3
import plotly.express as px
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURATION
# ==========================================
print("1️⃣ Chargement de la configuration...")
load_dotenv()

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID") or os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY") or os.getenv("AWS_SECRET_KEY")
BUCKET_NAME = os.getenv("BUCKET_NAME")

DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS") or os.getenv("DB_PASSWORD")

FILENAME = "weather_final.csv"

1️⃣ Chargement de la configuration...


In [2]:
# ==========================================
# 2. UPLOAD VERS S3
# ==========================================
print(f"\n2️⃣ Envoi de '{FILENAME}' vers S3...")
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="eu-west-3"
)
s3 = session.client('s3')

try:
    s3.upload_file(FILENAME, BUCKET_NAME, FILENAME)
    print(f"✅ Upload S3 réussi : s3://{BUCKET_NAME}/{FILENAME}")
except Exception as e:
    print(f"⚠️ Erreur Upload S3 : {e}")


2️⃣ Envoi de 'weather_final.csv' vers S3...
✅ Upload S3 réussi : s3://projet-booking-toso/weather_final.csv


In [3]:
# ==========================================
# 3. ALIMENTATION DE LA BASE SQL (RDS)
# ==========================================
print("\n3️⃣ Alimentation de la base SQL (RDS)...")

try:
    df_weather = pd.read_csv(FILENAME)
    
    db_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}?sslmode=require"
    engine = create_engine(db_url)
    
    df_weather.to_sql('weather', engine, if_exists='replace', index=False)
    print(f"✅ Table 'weather' mise à jour avec {len(df_weather)} villes.")

except Exception as e:
    print(f"❌ Erreur RDS : {e}")


3️⃣ Alimentation de la base SQL (RDS)...
✅ Table 'weather' mise à jour avec 35 villes.


In [4]:
# ==========================================
# 4. ANALYSE & CARTES
# ==========================================
print("\n🏆 GÉNÉRATION DES CARTES...")

query = """
SELECT 
    h.name as "Hôtel",
    h.city as "Ville",
    h.rating as "Note",
    h.price as "Prix",
    h.latitude as "Lat",
    h.longitude as "Lon",
    w.temperature_moyenne as "Température",
    w.pluie_totale_mm as "Pluie"
FROM hotels h
JOIN weather w ON h.city = w.ville
WHERE w.pluie_totale_mm < 5
  AND h.rating >= 8.0
ORDER BY h.rating DESC;
"""

try:
    df = pd.read_sql(query, engine)
    
    # Conversion types
    df['Lat'] = df['Lat'].astype(float)
    df['Lon'] = df['Lon'].astype(float)
    
    print(f"✅ Données récupérées : {len(df)} hôtels éligibles.")

    # CARTE 1 : TOP 5 VILLES
    top_cities = df.groupby('Ville').agg({
        'Note': 'mean',
        'Température': 'mean',
        'Lat': 'mean',
        'Lon': 'mean',
        'Hôtel': 'count'
    }).reset_index().sort_values(by='Note', ascending=False).head(5)

    print(f"📍 Top 5 Villes : {', '.join(top_cities['Ville'].tolist())}")

    fig1 = px.scatter_mapbox(
        top_cities, 
        lat="Lat", lon="Lon",
        color="Température", 
        size="Note", 
        hover_name="Ville",
        hover_data={"Température": True, "Note": True, "Hôtel": True},
        color_continuous_scale="Thermal",
        size_max=25, zoom=5, mapbox_style="open-street-map",
        title="TOP 5 Destinations : Le meilleur compromis Qualité / Météo"
    )
    fig1.show()

    # CARTE 2 : TOP 20 HÔTELS
    top_hotels = df.head(20)
    
    fig2 = px.scatter_mapbox(
        top_hotels, 
        lat="Lat", lon="Lon",
        color="Note", 
        size="Prix", 
        hover_name="Hôtel",
        hover_data=["Ville", "Température", "Prix"],
        color_continuous_scale="Tealgrn", 
        size_max=15, zoom=6, mapbox_style="open-street-map",
        title="TOP 20 Hôtels : Luxe et Soleil garanti"
    )
    fig2.show()

except Exception as e:
    print(f"❌ Erreur lors de l'analyse : {e}")


🏆 GÉNÉRATION DES CARTES...
✅ Données récupérées : 98 hôtels éligibles.
📍 Top 5 Villes : Collioure, Aigues Mortes, Saintes Maries de la mer, Uzes, Lyon
